<a href="https://colab.research.google.com/github/sukhadaghewari/tropicalcyclone-ri-bench/blob/main/notebooks/06_combine_satellite_images.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Combined satellite pictures: the latest 8 images in one picture

For every case (storm observation), this notebook makes **one picture** from the mid crops (about 1,170 km across) of the storm's latest 8 observations up to and including t:

- a 2 × 4 grid, read left to right, top row then bottom row: **oldest at top left, the case's own image (t, "now") always at bottom right**;
- under every box a label: `hours_since_start = x` (the same number the prompt's history uses for that observation), or `before first observation` when the storm has fewer than 8 observations so far;
- boxes with no image (before the storm's first observation, or an image that wasn't made) are white with grey hatching and say "no image";
- nothing else is drawn: no title, colour bar, time or place. Explain the grey scale and magenta in the prompt.

Every picture is 1042 × 578 px, small enough that none of the models we use shrinks it.

It needs the images from `04_download_satellite_images.ipynb` (`images/mid/` and `images/manifest.csv`). Pictures go to `images/combined/<case_id>_combined.png`, with a summary in `images/combined_manifest.csv`.

## 1. Setup

Works out the repo folder and makes `src/` importable, as in the other notebooks. In Colab it first clones this repo.

In [ ]:
# Setup: in Colab, clone the repo and move into it. Then put src/ on the import path.
import os, subprocess, sys
from pathlib import Path

REPO = "tropicalcyclone-ri-bench"
if "google.colab" in sys.modules:
    try:
        os.getcwd()
    except FileNotFoundError:              # the folder we were in was deleted (e.g. an old clone): start again
        os.chdir("/content")
if "google.colab" in sys.modules and os.path.basename(os.getcwd()) != REPO:
    if not os.path.exists(REPO):
        subprocess.run(["git", "clone", f"https://github.com/sukhadaghewari/{REPO}.git"], check=True)
    os.chdir(REPO)
if "google.colab" in sys.modules:
    # A clone from an earlier run survives "Restart session": bring it up to date with GitHub.
    if subprocess.run(["git", "pull", "--ff-only"]).returncode != 0:
        print("WARNING: could not update the repo with git pull; using the files as they are.")

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()   # repo root, locally or in Colab
sys.path.insert(0, str(ROOT / "src"))

## 2. Settings

**Where the images are.** By default in `model_inputs/images/` in the repo. In Colab, set `USE_DRIVE = True` to use the images saved to Google Drive by notebook 04 (the pictures are saved there too). Set `LIMIT` to a small number to try a few cases first (the summary file is then not written). Every run rebuilds all pictures, so they always match the current images and manifest.

**The picture.** 8 mid crops in 2 rows of 4, each resized to `PANEL` × `PANEL` pixels (a mid crop has only about 151 real GridSat pixels across, so 256 keeps all its detail), with a `LABEL_H`-pixel strip under each for its label and `GAP` pixels between boxes. The label font, DejaVu Sans, comes with matplotlib, so pictures look the same locally and in Colab.

In [ ]:
import pandas as pd
from matplotlib import font_manager
from PIL import Image, ImageDraw, ImageFont
from tqdm.auto import tqdm

from prompts import _clean, image_relpath, read_records

# ---- cases and images ---------------------------------------------------------------
CASE_FILES = [
    ROOT / "model_inputs" / "ri_prediction" / "dev" / "dev_2016_20ri_20nori_text_only.jsonl.gz",
    ROOT / "model_inputs" / "ri_prediction" / "test" / "test_2017-2025_50ri_50nori_text_only.jsonl.gz",
]
USE_DRIVE = False          # True: images are in Google Drive (Colab)
DRIVE_DIR = "wacv2027_output"   # folder inside MyDrive, as in notebook 04
LIMIT = None               # e.g. 20 to try the first 20 cases only

# ---- the picture ----------------------------------------------------------------------
CROP = "mid"               # about 1,170 km across
N_IMAGES = 8               # the case's own image plus up to 7 earlier ones
N_COLS = 4
PANEL = 256                # px per image
LABEL_H = 30               # px of white under each image for its label
GAP = 6                    # px of white between boxes
FONT = ImageFont.truetype(font_manager.findfont("DejaVu Sans"), 18)

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    out_dir = Path("/content/drive/MyDrive") / DRIVE_DIR
else:
    out_dir = ROOT / "model_inputs"

images_dir = out_dir / "images"
assert (images_dir / "manifest.csv").exists(), f"no images/manifest.csv in {out_dir}: run 04_download_satellite_images.ipynb first"
(images_dir / "combined").mkdir(exist_ok=True)
print("Using images in", images_dir)

## 3. Functions

A box for one image, an empty box ("no image", white with grey hatching), and the full picture from 8 boxes.

In [ ]:
def hours_label(hours):
    return f"hours_since_start = {_clean(hours)}"          # written exactly as in the prompt's history


def placeholder(text="no image"):
    """White box with light grey hatching and a short message, clearly not a satellite image."""
    im = Image.new("RGB", (PANEL, PANEL), "white")
    d = ImageDraw.Draw(im)
    for x in range(-PANEL, PANEL, 16):
        d.line([(x, PANEL), (x + PANEL, 0)], fill=(205, 205, 205), width=1)
    d.rectangle([0, 0, PANEL - 1, PANEL - 1], outline=(150, 150, 150))
    box = d.textbbox((PANEL // 2, PANEL // 2), text, font=FONT, anchor="mm")
    d.rectangle([box[0] - 8, box[1] - 6, box[2] + 8, box[3] + 6], fill="white")
    d.text((PANEL // 2, PANEL // 2), text, fill=(70, 70, 70), font=FONT, anchor="mm")
    return im


def combine(panels):
    """panels: N_IMAGES items, oldest first, each (PIL image or None, label text)."""
    n_rows = -(-N_IMAGES // N_COLS)
    width = N_COLS * PANEL + (N_COLS - 1) * GAP
    height = n_rows * (PANEL + LABEL_H) + (n_rows - 1) * GAP
    out = Image.new("RGB", (width, height), "white")
    d = ImageDraw.Draw(out)
    for i, (image, label) in enumerate(panels):
        x = (i % N_COLS) * (PANEL + GAP)
        y = (i // N_COLS) * (PANEL + LABEL_H + GAP)
        out.paste(image if image is not None else placeholder(), (x, y))
        d.text((x + PANEL // 2, y + PANEL + LABEL_H // 2), label, fill="black", font=FONT, anchor="mm")
    return out

## 4. Build the pictures

For the case at observation k, the boxes hold observations k-7 ... k of the same storm, oldest first; the case's own image is always the last box. An image is used only if the manifest says it was made (`status == "ok"`). Each image is read once per storm, then reused for the up to 8 pictures it appears in.

Takes about a minute on a local disk, longer on Google Drive. Existing pictures are overwritten. The summary says how many cases have no image at t (box 8 shows "no image") and how many real images the pictures show.

In [ ]:
cases = pd.DataFrame([r for f in CASE_FILES for r in read_records(f)])
cases = cases[["case_id", "sid", "obs_index", "hours_since_start"]].sort_values(["sid", "obs_index"])
assert cases["case_id"].is_unique, "a case_id appears twice across the case files"
for sid, storm in cases.groupby("sid"):
    assert storm["obs_index"].tolist() == list(range(len(storm))), f"{sid}: observations are not 0, 1, 2, ..."

manifest = pd.read_csv(images_dir / "manifest.csv", dtype={"id": str})
made = set(manifest.loc[manifest["status"] == "ok", "id"])

todo = cases.head(LIMIT) if LIMIT else cases
rows = []
for sid, storm in tqdm(todo.groupby("sid", sort=False), unit="storm"):
    storm_all = cases[cases["sid"] == sid].set_index("obs_index")
    cache = {}                                   # each image is used by up to 8 pictures: read it once

    def panel(k):
        if k not in cache:
            cid = storm_all.at[k, "case_id"]
            path = out_dir / image_relpath(cid, CROP)
            cache[k] = (Image.open(path).convert("RGB").resize((PANEL, PANEL), Image.LANCZOS)
                        if cid in made and path.exists() else None)
        return cache[k]

    for case in storm.itertuples():
        out_path = out_dir / image_relpath(case.case_id, "combined")
        shown = list(range(max(0, case.obs_index - N_IMAGES + 1), case.obs_index + 1))
        rows.append({"case_id": case.case_id, "n_observations": len(shown),
                     "n_images": sum(panel(k) is not None for k in shown),
                     "image_at_t": panel(case.obs_index) is not None})
        slots = [(None, "before first observation")] * (N_IMAGES - len(shown))
        slots += [(panel(k), hours_label(storm_all.at[k, "hours_since_start"])) for k in shown]
        combine(slots).save(out_path)

summary = pd.DataFrame(rows)
if not LIMIT:                                    # a trial run must not replace the full summary
    summary.to_csv(images_dir / "combined_manifest.csv", index=False)
print(f"{len(summary)} pictures in {images_dir / 'combined'}")
print(f"image at t missing: {int((~summary['image_at_t']).sum())} cases")
print("images shown per picture:", summary["n_images"].value_counts().sort_index().to_dict())
summary.head()

## 5. Look at a few pictures

Three cases from the same storm: its first observation (only box 8 has an image), one with a few earlier images, and its latest, with all 8 boxes filled if the storm is long enough.

In [ ]:
import matplotlib.pyplot as plt

sid = summary["case_id"].str.rsplit("_", n=1).str[0].value_counts().index[0]   # the storm with the most cases built
storm = summary[summary["case_id"].str.startswith(sid + "_")]
examples = storm["case_id"].iloc[[0, min(3, len(storm) - 1), -1]].unique()

for cid in examples:
    fig, ax = plt.subplots(figsize=(12, 6.6))
    ax.imshow(Image.open(out_dir / image_relpath(cid, "combined")))
    ax.set_title(cid)
    ax.axis("off")
    plt.show()

## Notes

- **Box 8 is always now:** the image of the case's own observation (time t). The boxes before it are the same storm's earlier observations, 6 hours apart in our data, oldest first.
- **Never show the model a file name:** picture names contain the case ID. Send the model the picture itself, not its path.
- **Using the pictures:** join a case file with `images/combined_manifest.csv` on `case_id`. Its columns are `n_observations` (boxes with a real observation), `n_images` (boxes with an image) and `image_at_t` (False when box 8 says "no image").